# L12d Lab: Exploration, Seeds, and Defensible RL Results

> **Learning objectives**
>
> - Compare an exploration schedule with a prematurely greedy learner.
> - Repeat evaluation under controlled seeds.
> - Report policy agreement and returns rather than one attractive trajectory.
> - State the limits of a small tabular demonstration.


## Setup

Run the local setup cell first. It activates the pinned course environment, loads every package used by this meeting, and includes the `Week12Core` module from [`../src/Week12Core.jl`](../src/Week12Core.jl), which provides the functions called below.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes the meeting's local source. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


The setup cell completed, so the environment and this lab's local source are loaded. The cell below builds `world::NamedTuple` and the exact `reference::NamedTuple` from value iteration, then trains two seeded Q-learning agents that differ only in exploration schedule, `exploring::NamedTuple` and `premature::NamedTuple`. The comparison is `results::DataFrame`.


In [2]:
world = learning_gridworld()
reference = value_iteration_reference(world)
exploring = q_learning(world; episodes = 12_000, epsilon_start = 1.0,
    epsilon_min = 0.03, epsilon_decay = 0.9993, seed = 5800)
premature = q_learning(world; episodes = 12_000, epsilon_start = 0.05,
    epsilon_min = 0.0, epsilon_decay = 0.99, seed = 5800)

results = DataFrame(
    learner = ["scheduled exploration", "prematurely greedy"],
    policy_agreement = [policy_agreement(reference.policy, exploring.policy, world.terminal),
        policy_agreement(reference.policy, premature.policy, world.terminal)],
    evaluated_return = [evaluate_policy(world, exploring.policy; seed = 9101).mean_return,
        evaluate_policy(world, premature.policy; seed = 9101).mean_return],
)
pretty_table(results)


┌───────────────────────┬──────────────────┬──────────────────┐
│               learner │ policy_agreement │ evaluated_return │
│                String │          Float64 │          Float64 │
├───────────────────────┼──────────────────┼──────────────────┤
│ scheduled exploration │         0.888889 │         0.533654 │
│    prematurely greedy │         0.666667 │         0.501451 │
└───────────────────────┴──────────────────┴──────────────────┘


In [3]:
@test results.policy_agreement[1] >= results.policy_agreement[2]
@test results.evaluated_return[1] >= results.evaluated_return[2] - 1e-10
@test q_learning(world; episodes = 100, seed = 9).Q == q_learning(world; episodes = 100, seed = 9).Q
:exploration_reproducibility_verified


:exploration_reproducibility_verified

## Course integration

The Fall sequence now closes a full computational loop: represent state, compute or learn a policy, validate it against explicit contracts, and interpret the result under model and data limitations. Deep function approximation remains Spring material.
